In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("NULL_skew") \
    .master("local[*]") \
    .getOrCreate()

----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 10960)
Traceback (most recent call last):
  File "D:\Python\Python38\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "D:\Python\Python38\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "D:\Python\Python38\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "D:\Python\Python38\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "e:\workspace\python_script\.venv\lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "e:\workspace\python_script\.venv\lib\site-packages\pyspark\accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "e:\workspace\python_script\.venv\lib\site-packages\pyspark\accumulators.py", line 271,

In [ ]:
# 倾斜数据：10万条 NULL
data_null = [(None, f"user_{i}") for i in range(100000)]

# 正常数据：10条
data_normal = [(str(i), f"user_{i}") for i in range(1, 11)]

# 合并
data = data_null + data_normal

df = spark.createDataFrame(data, ["key", "info"])
df.show()


+----+-------+
| key|   info|
+----+-------+
|NULL| user_0|
|NULL| user_1|
|NULL| user_2|
|NULL| user_3|
|NULL| user_4|
|NULL| user_5|
|NULL| user_6|
|NULL| user_7|
|NULL| user_8|
|NULL| user_9|
|NULL|user_10|
|NULL|user_11|
|NULL|user_12|
|NULL|user_13|
|NULL|user_14|
|NULL|user_15|
|NULL|user_16|
|NULL|user_17|
|NULL|user_18|
|NULL|user_19|
+----+-------+
only showing top 20 rows



In [6]:
df.tail(11)

[Row(key=None, info='user_99999'),
 Row(key='1', info='user_1'),
 Row(key='2', info='user_2'),
 Row(key='3', info='user_3'),
 Row(key='4', info='user_4'),
 Row(key='5', info='user_5'),
 Row(key='6', info='user_6'),
 Row(key='7', info='user_7'),
 Row(key='8', info='user_8'),
 Row(key='9', info='user_9'),
 Row(key='10', info='user_10')]

In [8]:
df.groupBy("key") \
  .agg(count("*").alias("cnt")) \
  .orderBy(col("cnt").desc()) \
  .show()

+----+------+
| key|   cnt|
+----+------+
|NULL|100000|
|   7|     1|
|   3|     1|
|   8|     1|
|   5|     1|
|   6|     1|
|   9|     1|
|   1|     1|
|  10|     1|
|   4|     1|
|   2|     1|
+----+------+



In [10]:
df_fixed = df.withColumn(
    "new_key",
    when(
        col("key").isNull(),
        concat(lit("null_"), (rand() * 20).cast("int"))  # 随机 0~19
    ).otherwise(col("key"))
)
df_fixed.tail(50)

[Row(key=None, info='user_99960', new_key='null_17'),
 Row(key=None, info='user_99961', new_key='null_1'),
 Row(key=None, info='user_99962', new_key='null_5'),
 Row(key=None, info='user_99963', new_key='null_0'),
 Row(key=None, info='user_99964', new_key='null_2'),
 Row(key=None, info='user_99965', new_key='null_14'),
 Row(key=None, info='user_99966', new_key='null_4'),
 Row(key=None, info='user_99967', new_key='null_2'),
 Row(key=None, info='user_99968', new_key='null_16'),
 Row(key=None, info='user_99969', new_key='null_13'),
 Row(key=None, info='user_99970', new_key='null_2'),
 Row(key=None, info='user_99971', new_key='null_10'),
 Row(key=None, info='user_99972', new_key='null_14'),
 Row(key=None, info='user_99973', new_key='null_10'),
 Row(key=None, info='user_99974', new_key='null_17'),
 Row(key=None, info='user_99975', new_key='null_6'),
 Row(key=None, info='user_99976', new_key='null_4'),
 Row(key=None, info='user_99977', new_key='null_0'),
 Row(key=None, info='user_99978', new_

In [12]:
dim_data = [("1", "男"), ("2", "女"), ("3", "未知")]
df_dim = spark.createDataFrame(dim_data, ["key", "gender"])
df_dim.show()

+---+------+
|key|gender|
+---+------+
|  1|    男|
|  2|    女|
|  3|  未知|
+---+------+



In [14]:
# 小表也要同样规则处理
df_dim_expand = df_dim.withColumn(
    "salt", explode(array(*[lit(i) for i in range(20)]))
).withColumn(
    "new_key",
    when(col("key").isNull(), 
         concat(lit("null_"), col("salt"))
    ).otherwise(col("key"))
)
df_dim_expand.show()

+---+------+----+-------+
|key|gender|salt|new_key|
+---+------+----+-------+
|  1|    男|   0|      1|
|  1|    男|   1|      1|
|  1|    男|   2|      1|
|  1|    男|   3|      1|
|  1|    男|   4|      1|
|  1|    男|   5|      1|
|  1|    男|   6|      1|
|  1|    男|   7|      1|
|  1|    男|   8|      1|
|  1|    男|   9|      1|
|  1|    男|  10|      1|
|  1|    男|  11|      1|
|  1|    男|  12|      1|
|  1|    男|  13|      1|
|  1|    男|  14|      1|
|  1|    男|  15|      1|
|  1|    男|  16|      1|
|  1|    男|  17|      1|
|  1|    男|  18|      1|
|  1|    男|  19|      1|
+---+------+----+-------+
only showing top 20 rows



In [17]:
result = df_fixed.join(broadcast(df_dim_expand), on="new_key",how='left')
result.show()

+-------+----+-------+----+------+----+
|new_key| key|   info| key|gender|salt|
+-------+----+-------+----+------+----+
|null_12|NULL| user_0|NULL|  NULL|NULL|
|null_10|NULL| user_1|NULL|  NULL|NULL|
|null_10|NULL| user_2|NULL|  NULL|NULL|
| null_3|NULL| user_3|NULL|  NULL|NULL|
| null_1|NULL| user_4|NULL|  NULL|NULL|
| null_9|NULL| user_5|NULL|  NULL|NULL|
|null_14|NULL| user_6|NULL|  NULL|NULL|
|null_11|NULL| user_7|NULL|  NULL|NULL|
| null_5|NULL| user_8|NULL|  NULL|NULL|
|null_12|NULL| user_9|NULL|  NULL|NULL|
| null_3|NULL|user_10|NULL|  NULL|NULL|
|null_15|NULL|user_11|NULL|  NULL|NULL|
| null_2|NULL|user_12|NULL|  NULL|NULL|
| null_7|NULL|user_13|NULL|  NULL|NULL|
| null_6|NULL|user_14|NULL|  NULL|NULL|
|null_16|NULL|user_15|NULL|  NULL|NULL|
|null_10|NULL|user_16|NULL|  NULL|NULL|
|null_17|NULL|user_17|NULL|  NULL|NULL|
|null_14|NULL|user_18|NULL|  NULL|NULL|
|null_15|NULL|user_19|NULL|  NULL|NULL|
+-------+----+-------+----+------+----+
only showing top 20 rows



In [24]:
result=result.drop(df_dim_expand['key'])
result.show()

+-------+-------+------+----+
|new_key|   info|gender|salt|
+-------+-------+------+----+
|null_12| user_0|  NULL|NULL|
|null_10| user_1|  NULL|NULL|
|null_10| user_2|  NULL|NULL|
| null_3| user_3|  NULL|NULL|
| null_1| user_4|  NULL|NULL|
| null_9| user_5|  NULL|NULL|
|null_14| user_6|  NULL|NULL|
|null_11| user_7|  NULL|NULL|
| null_5| user_8|  NULL|NULL|
|null_12| user_9|  NULL|NULL|
| null_3|user_10|  NULL|NULL|
|null_15|user_11|  NULL|NULL|
| null_2|user_12|  NULL|NULL|
| null_7|user_13|  NULL|NULL|
| null_6|user_14|  NULL|NULL|
|null_16|user_15|  NULL|NULL|
|null_10|user_16|  NULL|NULL|
|null_17|user_17|  NULL|NULL|
|null_14|user_18|  NULL|NULL|
|null_15|user_19|  NULL|NULL|
+-------+-------+------+----+
only showing top 20 rows



In [ ]:
result.groupBy("key", "gender") \
      .count() \
      .orderBy(col("count").desc()) \
      .show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `key` cannot be resolved. Did you mean one of the following? [`info`, `new_key`, `salt`, `gender`].;
'Project ['key, info#1, gender#60, 'join_key]
+- Project [new_key#51, info#1, gender#60, salt#100]
   +- Project [new_key#51, key#0, info#1, key#59, gender#60, salt#100]
      +- Join LeftOuter, (new_key#51 = new_key#104)
         :- Project [key#0, info#1, CASE WHEN isnull(key#0) THEN concat(null_, cast(cast((rand(1756311445303142097) * cast(20 as double)) as int) as string)) ELSE key#0 END AS new_key#51]
         :  +- LogicalRDD [key#0, info#1], false
         +- ResolvedHint (strategy=broadcast)
            +- Project [key#59, gender#60, salt#100, CASE WHEN isnull(key#59) THEN concat(null_, cast(salt#100 as string)) ELSE key#59 END AS new_key#104]
               +- Project [key#59, gender#60, salt#100]
                  +- Generate explode(array(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 1